# Triplet Neural Network

This notebook demonstrates how to train one `Triplet Neural Network` for image similarity computation. For this, the `Oxford Flowers` dataset is used, but customized for the triplet network training.

# Import libraries

In [1]:
from typing import Any
from collections import defaultdict
from collections.abc import Iterator
import multiprocessing
import random
import datetime
import os

import torch
from torch.utils.data import DataLoader, Dataset, Sampler
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms

from pyvisim.neural_networks import TripletNeuralNetwork
from pyvisim.datasets import OxfordFlowerDataset

# Declare the dataset

A labeled batch of images goes through the shared-weight tower in a single pass, and the loss mines the (anchor, positive, negative) triplets from that batch itself.

How a batch is drawn is very important. If drawn at random, a dataset with 1000 classes will yield very few positive pairs if the batch size is not large enough, and the network will learn very slowly. 

Hence, the  `PKBatchSampler` is implemented. It draws `classes_per_batch` classes and `samples_per_class` images of each of them. `batch_size=classes_per_batch * samples_per_class`. With this, each batch is guaranteed to contain `samples_per_class - 1 ` positives and `batch_size - samples_per_class` negatives (assuming every sampled class has at least `samples_per_class` examples) for each image.

The resulting batch is passed through the network, producing a `batch_size x batch_size` matrix of distances. The `TripletLoss` then mines the (hardest) triplets from this matrix and computes the loss. The network is trained to minimize this loss.

In [2]:
class OxfordLabeledImages(Dataset[tuple[torch.Tensor, torch.Tensor]]):
    """
    Labeled single images on top of :class:`OxfordFlowerDataset`.

    Each item is an ``(image, label)`` pair. The labels are also exposed as a
    plain list, so the batch sampler can group the indices by class without
    decoding a single image.

    :param purpose: Dataset split to draw from (e.g. ``"train"``).
    :param transform: Transform applied to each image.
    """

    def __init__(self, purpose: str, transform: transforms.Compose) -> None:
        self._base = OxfordFlowerDataset(transform=None, purpose=purpose)
        self.transform = transform
        self.labels: list[int] = list(self._base.labels)

    def __len__(self) -> int:
        return len(self._base)

    def _get_tensor(self, idx: int) -> torch.Tensor:
        image, _, _ = self._base[idx]
        tensor: torch.Tensor = self.transform(image)
        return tensor

    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return self._get_tensor(idx), label

## The P-K batch sampler

`fixed_batches=True` restarts the sampler from the same seed on every epoch. The validation loss is then measured on the same triplets each time and stays comparable across epochs.

In [3]:
class PKBatchSampler(Sampler[list[int]]):
    """
    P-K batch sampler: ``classes_per_batch`` classes, ``samples_per_class`` images each.

    Classes with fewer than ``samples_per_class`` images are sampled with
    replacement, so every batch has exactly the same size.

    :param labels: Class label of each dataset index.
    :param classes_per_batch: Number of distinct classes per batch (P).
    :param samples_per_class: Number of images drawn per sampled class (K).
    :param seed: Seed of the internal random generator.
    :param fixed_batches: If ``True``, every epoch yields the same batches.
    :raises ValueError: If the split holds fewer classes than
        ``classes_per_batch``, or if fewer than two classes are available.
    """

    def __init__(
        self,
        labels: list[int],
        classes_per_batch: int,
        samples_per_class: int,
        seed: int,
        fixed_batches: bool = False,
    ) -> None:
        # Labels are read directly, as indexing the base dataset would decode
        # every image just to group the indices by class.
        self._label_to_indices: dict[int, list[int]] = defaultdict(list)
        for idx, label in enumerate(labels):
            self._label_to_indices[label].append(idx)

        self._labels_list = list(self._label_to_indices.keys())
        if len(self._labels_list) < 2:
            raise ValueError("Need >= 2 classes for triplet training.")
        if len(self._labels_list) < classes_per_batch:
            raise ValueError(
                f"Need >= {classes_per_batch} classes for a batch of "
                f"{classes_per_batch} classes, got {len(self._labels_list)}."
            )

        self.classes_per_batch = classes_per_batch
        self.samples_per_class = samples_per_class
        self.fixed_batches = fixed_batches
        self._seed = seed
        self._rng = random.Random(seed)
        self._num_batches = len(labels) // (classes_per_batch * samples_per_class)

    def __len__(self) -> int:
        return self._num_batches

    def _sample_class(self, label: int, rng: random.Random) -> list[int]:
        indices = self._label_to_indices[label]
        if len(indices) >= self.samples_per_class:
            return rng.sample(indices, self.samples_per_class)
        return rng.choices(indices, k=self.samples_per_class)

    def __iter__(self) -> Iterator[list[int]]:
        rng = random.Random(self._seed) if self.fixed_batches else self._rng
        for _ in range(self._num_batches):
            labels = rng.sample(self._labels_list, self.classes_per_batch)
            yield [idx for label in labels for idx in self._sample_class(label, rng)]

# Training seeds

In [4]:
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"

# The batch size is classes_per_batch * samples_per_class = 64. The smallest
# split of Oxford Flowers still holds 10 images per class, so K = 4 fits into
# every class without resampling.
CONFIG_DATALOADER: dict[str, Any] = {
    "classes_per_batch": 16,
    "samples_per_class": 4,
    "num_workers": 4,
}

def seed_worker(worker_id: int) -> None:
    random.seed(torch.initial_seed() % 2**32)

# Transforms

The transforms below match the ones used in the pre-trained backbone (*ResNet-18*, in the default configuration, which is also the configuration below).

In [5]:
_NORMALIZE = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)

TRAIN_TF = transforms.Compose(
    [
        transforms.ToPILImage(),
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.RandomApply(
            [
                transforms.RandomChoice([
                    transforms.RandomHorizontalFlip(p=1.0),
                    transforms.RandomRotation(15),
                ]),
            ],
            p=0.5,
        ),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
        transforms.ToTensor(),
        _NORMALIZE,
    ]
)

VAL_TF = transforms.Compose(
    [
        transforms.ToPILImage(),
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        _NORMALIZE,
    ]
)

# Multiprocessing configs

This was necessary on `Linux` and `Python 3.14+`. You can comment it out if it caused you trouble.

In [6]:
# Python 3.14 made "forkserver" the default start method on Linux. A forkserver
# worker re-imports the payload in a fresh interpreter, so it cannot resolve
# classes defined in a notebook (they live in ``__main__``). Forking inherits the
# interpreter state instead, which keeps ``OxfordLabeledImages`` usable.
MP_CONTEXT = "fork" if "fork" in multiprocessing.get_all_start_methods() else None

def make_loader(
    dataset: OxfordLabeledImages,
    fixed_batches: bool,
) -> DataLoader[tuple[torch.Tensor, torch.Tensor]]:
    num_workers = CONFIG_DATALOADER["num_workers"] if MP_CONTEXT is not None else 0
    # The sampler already shuffles, so the loader takes no "shuffle" flag.
    batch_sampler = PKBatchSampler(
        dataset.labels,
        classes_per_batch=CONFIG_DATALOADER["classes_per_batch"],
        samples_per_class=CONFIG_DATALOADER["samples_per_class"],
        seed=SEED,
        fixed_batches=fixed_batches,
    )
    return DataLoader(
        dataset,
        batch_sampler=batch_sampler,
        num_workers=num_workers,
        pin_memory=True,
        worker_init_fn=seed_worker,
        multiprocessing_context=MP_CONTEXT if num_workers > 0 else None,
        persistent_workers=num_workers > 0,
    )

# Part 1: Semi-hard online mining

Introduced by Hoffer and Ailon in 2014 and popularized by *FaceNet* (Schroff et al., 2015), the triplet network learns the embedding space from relative comparisons instead of absolute ones: an anchor has to be closer to a positive image (same class) than to a negative one (different class), by at least a margin.

The three branches of the classical drawing are one single network. Anchor, positive and negative all pass through the same backbone (ResNet-18 here) and the same projection head, and the resulting embeddings are L2-normalized.

`TripletLoss` is defined as follows:

$$
L(a, p, n) = \max\Big(0, \, d(a, p) - d(a, n) + m\Big)
$$

Whereas:

- *a*, *p*, *n*: anchor, positive (same class as the anchor) and negative (different class)
- *d*: the (optionally squared) Euclidean distance between two embeddings
- *m*: margin: the minimum gap enforced between the positive and the negative distance

The triplets are mined online, meaning that they are picked from the labeled batch during the forward pass. `mining="semi_hard"` is FaceNet's strategy: for every positive pair, the closest negative that is still farther away than the positive is chosen. Such negatives already violate the margin and hence produce a gradient, without being so hard that the embedding collapses.

In [7]:
from pyvisim.neural_networks.losses import TripletLoss

# Train parameters

In [8]:
CONFIG_SEMI_HARD: dict[str, Any] = {
    "embedding_dim": 64,
    "margin": 0.2,
    "mining": "semi_hard",
    "squared": True,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "num_epochs": 20,
    "checkpoint_dir": "checkpoints",
    "log_every_n": 50,
    "gamma": 0.8,
}

## Some train helpers

Both parts of this notebook train the very same network with the very same loop. Only the mining configuration changes, so the helpers below take the config and the run name as parameters.

In [9]:
def _run_epoch_triplet(
    model: TripletNeuralNetwork,
    loader: DataLoader[tuple[torch.Tensor, torch.Tensor]],
    criterion: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    is_train: bool,
    epoch: int,
    writer: SummaryWriter,
    config: dict[str, Any],
) -> float:
    model.train() if is_train else model.eval()
    total_loss = 0.0
    ctx = torch.enable_grad() if is_train else torch.no_grad()

    with ctx:
        for step, (images, labels) in enumerate(loader):
            images = images.to(device)
            labels = labels.to(device)

            # One shared-weight pass over the whole batch. The triplets are
            # mined from these embeddings by the loss itself.
            embeddings = model(images)
            loss = criterion(embeddings, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                # Gradient clipping to prevent exploding gradients.
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss += loss.item()

            if is_train and (step + 1) % config["log_every_n"] == 0:
                print(
                    f"  Epoch {epoch:03d} | step {step + 1:04d} "
                    f"| loss {total_loss / (step + 1):.4f}"
                )

            loss_str = "Loss/Train" if is_train else "Loss/Validation"
            global_step = (epoch - 1) * len(loader) + step
            writer.add_scalar(loss_str, loss.item(), global_step=global_step)

    return total_loss / len(loader)


def train_triplet(
    model: TripletNeuralNetwork,
    criterion: torch.nn.Module,
    config: dict[str, Any],
    run_name: str,
) -> None:

    session_id = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    writer = SummaryWriter(log_dir=f"runs/triplet/session_{session_id}_{run_name}")

    try:
        os.makedirs(config["checkpoint_dir"], exist_ok=True)

        train_ds = OxfordLabeledImages("train", transform=TRAIN_TF)
        val_ds = OxfordLabeledImages("validation", transform=VAL_TF)

        train_loader = make_loader(train_ds, fixed_batches=False)
        val_loader = make_loader(val_ds, fixed_batches=True)

        batch_size = (
            CONFIG_DATALOADER["classes_per_batch"]
            * CONFIG_DATALOADER["samples_per_class"]
        )
        print(f"Train images: {len(train_ds):,}  |  Val images: {len(val_ds):,}")
        print(f"Flower classes: {len(set(train_ds.labels))}")
        print(f"Batch size: {batch_size}  |  Batches per epoch: {len(train_loader)}")

        optimizer = torch.optim.AdamW(
            model.parameters(), lr=config["lr"], weight_decay=config["weight_decay"]
        )
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=4, gamma=config.get('gamma', 0.8))

        best_val = float("inf")
        for epoch in range(1, config["num_epochs"] + 1):
            train_loss = _run_epoch_triplet(
                model, train_loader, criterion, optimizer, is_train=True, epoch=epoch, writer=writer, config=config
            )
            val_loss = _run_epoch_triplet(
                model, val_loader, criterion, optimizer, is_train=False, epoch=epoch, writer=writer, config=config
            )
            scheduler.step()

            print(
                f"Epoch {epoch:03d} | train {train_loss:.4f} | val {val_loss:.4f} "
                f"| lr {scheduler.get_last_lr()[0]:.2e}"
            )
            model.save_to_disk(f"{run_name}.embedder")
            if val_loss < best_val:
                best_val = val_loss
                model.save_to_disk(f"{run_name}_best.embedder")
                print(f"New best val loss: {best_val:.4f}")

        print(f"\nTraining complete. Best val loss: {best_val:.4f}")
    finally:
        writer.close()

In [ ]:
criterion = TripletLoss(
    margin=CONFIG_SEMI_HARD["margin"],
    mining=CONFIG_SEMI_HARD["mining"],
    squared=CONFIG_SEMI_HARD["squared"],
)
model_semi_hard = TripletNeuralNetwork(
    backbone="resnet18",
    embedding_dim=CONFIG_SEMI_HARD["embedding_dim"],
    device=device,
    pretrained_backbone=True,
)
train_triplet(model_semi_hard, criterion, CONFIG_SEMI_HARD, run_name="triplet_semi_hard")

Train images: 6,149  |  Val images: 1,020
Flower classes: 102
Batch size: 64  |  Batches per epoch: 96
  Epoch 001 | step 0050 | loss 0.0892
Epoch 001 | train 0.0661 | val 0.0415 | lr 1.00e-04
New best val loss: 0.0415
  Epoch 002 | step 0050 | loss 0.0293
Epoch 002 | train 0.0252 | val 0.0298 | lr 1.00e-04
New best val loss: 0.0298
  Epoch 003 | step 0050 | loss 0.0151
Epoch 003 | train 0.0146 | val 0.0255 | lr 1.00e-04
New best val loss: 0.0255
  Epoch 004 | step 0050 | loss 0.0107
Epoch 004 | train 0.0100 | val 0.0204 | lr 8.00e-05
New best val loss: 0.0204
  Epoch 005 | step 0050 | loss 0.0067
Epoch 005 | train 0.0059 | val 0.0191 | lr 8.00e-05
New best val loss: 0.0191
  Epoch 006 | step 0050 | loss 0.0043
Epoch 006 | train 0.0046 | val 0.0172 | lr 8.00e-05
New best val loss: 0.0172


# Load models after training

In [ ]:
model_semi_hard = TripletNeuralNetwork.load_from_disk("triplet_semi_hard_best.embedder")
model_semi_hard.eval()

# Visualize results

Now, we will pick three random images from the validation set, two of which are similar and one is dissimilar.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


def visualize_image_pair(image_a: np.ndarray, image_b: np.ndarray) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(image_a)
    axes[0].set_title("Image A")
    axes[0].axis("off")

    axes[1].imshow(image_b)
    axes[1].set_title("Image B")
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

test_dataset =  OxfordFlowerDataset(None, "test")
img_a = test_dataset[4][0]
img_b = test_dataset[5][0]
img_c = test_dataset[100][0]
print("Image shapes:\n")
print(f"Image A: {img_a.shape}")
print(f"Image B: {img_b.shape}")

visualize_image_pair(img_a, img_b)
visualize_image_pair(img_a, img_c)

# Compute similarity

As observed, the pair of the same class yields a similarity score close to 1, while the pair of different classes yields a similarity score should be much lower (at least below 0.0, or at best close to -1.0). The embeddings are L2-normalized, so the cosine similarity is simply their dot product.

In [ ]:
score_same_class = model_semi_hard.similarity_score(img_a, img_b).item()
score_different_class = model_semi_hard.similarity_score(img_a, img_c).item()

print(f"Similarity score (same class): {score_same_class:.4f}")
print(f"Similarity score (different class): {score_different_class:.4f}")

# Part 2: Batch-hard online mining

*In Defense of the Triplet Loss* (Hermans et al., 2017)'s approach keeps only the two extremes of every anchor inside the batch: its farthest positive and its closest negative. Fewer triplets contribute, but each of them carries a much stronger gradient. The same paper reports using Euclidean distance is more stable, while the squared distance made the optimization more prone to collapsing (all embeddings mapped to the same point), which is why `squared` is turned off here.

## NOTE

This is the very same network and the very same training loop as in part 1. Only the mined triplets differ, which is also why the P-K sampling matters even more here: an anchor without a positive in its batch contributes nothing at all to the batch-hard loss.

# Train parameters

In [ ]:
CONFIG_BATCH_HARD: dict[str, Any] = {
    "embedding_dim": 64,
    "margin": 0.5,
    "mining": "batch_hard",
    "squared": False,
    "lr": 1e-4,
    "weight_decay": 1e-4,
    "num_epochs": 20,
    "checkpoint_dir": "checkpoints",
    "log_every_n": 50,
    "gamma": 0.8,
}

In [ ]:
criterion = TripletLoss(
    margin=CONFIG_BATCH_HARD["margin"],
    mining=CONFIG_BATCH_HARD["mining"],
    squared=CONFIG_BATCH_HARD["squared"],
)
model_batch_hard = TripletNeuralNetwork(
    backbone="resnet18",
    embedding_dim=CONFIG_BATCH_HARD["embedding_dim"],
    device=device,
    pretrained_backbone=True,
)
train_triplet(model_batch_hard, criterion, CONFIG_BATCH_HARD, run_name="triplet_batch_hard")

# Load models after training

In [ ]:
model_batch_hard = TripletNeuralNetwork.load_from_disk("triplet_batch_hard_best.embedder")
model_batch_hard.eval()

# Visualize results

The same three images are reused, so the two mining strategies can be compared on identical inputs.

In [ ]:
print("Image shapes:\n")
print(f"Image A: {img_a.shape}")
print(f"Image B: {img_b.shape}")

visualize_image_pair(img_a, img_b)
visualize_image_pair(img_a, img_c)

# Compute similarity

Both networks produce L2-normalized embeddings, so the scores below live on the same scale as the ones of part 1 and can be read side by side.

In [ ]:
score_same_class = model_batch_hard.similarity_score(img_a, img_b).item()
score_different_class = model_batch_hard.similarity_score(img_a, img_c).item()

print(f"Similarity score (same class): {score_same_class:.4f}")
print(f"Similarity score (different class): {score_different_class:.4f}")

# References

[1] Hoffer, E., & Ailon, N. (2014). Deep Metric Learning Using Triplet
Network. https://arxiv.org/abs/1412.6622


[2] Schroff, F., Kalenichenko, D., & Philbin, J. (2015). FaceNet: A Unified
Embedding for Face Recognition and Clustering. In Proceedings of the 2015
IEEE Conference on Computer Vision and Pattern Recognition (CVPR),
815-823. https://doi.org/10.1109/CVPR.2015.7298682


[3] Hermans, A., Beyer, L., & Leibe, B. (2017). In Defense of the Triplet
Loss for Person Re-Identification. https://arxiv.org/abs/1703.07737